# 🧪 Smart Microscopy Exercise
## From Overview Imaging to Targeted Acquisition

In this exercise, you will simulate a **smart microscopy workflow** using real data of HeLa cells infected with bacteria.

Author: Hannah S. Heil, Lund University, hannah.heil@med.lu.se, July 2026
Documentation:
https://github.com/HannahSHeil/Smart-Microscopy-Exercise

---

### 🎯 Goal

Learn how microscopy can move from:

👉 **Passive acquisition** → **Adaptive, data-driven decisions**

---

### 🔁 Core Concept

**Acquire → Analyze → Decide → Adapt**

---

By the end of this exercise, you will:
- Segment cells using **AI (CellSAM)**
- Detect **bacterial infection**
- Select **cells of interest**
- Perform **targeted acquisition**
- Quantify **efficiency gains**




# Step 1: Ensure that you run on a GPU
!! User intervetion required !!:

Go to:
Runtime → Change runtime type → Hardware accelerator → GPU

In [ ]:
# @title Step 2 — Install Dependencies

!pip install cellpose segment-anything opencv-python scikit-image scipy



# Step 3: Restart runtime

 !! User intervetion required !!
Go to: Runtime → Restart runtime

In [ ]:
# @title Step 4 — Imports

import numpy as np
import matplotlib.pyplot as plt
from skimage import io
from scipy.ndimage import distance_transform_edt
import ipywidgets as widgets
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display

import random


from skimage import measure
from skimage.measure import label, regionprops_table
from skimage.morphology import remove_small_objects

import pandas as pd


from cellpose import models
import torch

print("GPU:", torch.cuda.is_available())



# Step 5: Are you running on a GPU?
Attention!!

If "GPU: False" please change the runtime type to GPU!


Go to:
Runtime → Change runtime type → Hardware accelerator → GPU

In [ ]:
# @title Step 6 — Mount your sample (Load Data from GitHub)

# Download files from Zenodo
!wget "https://zenodo.org/records/21064217/files/C1-highres.tif?download=1" -O dic-highres.tif
!wget "https://zenodo.org/records/21064217files/C2-highres.tif?download=1" -O actin-highres.tif
!wget "https://zenodo.org/records/21064217files/C3-highres.tif?download=1" -O bacteria-highres.tif

!wget "https://zenodo.org/records/21064217/files/C1-lowres.tif?download=1" -O dic-lowres.tif
!wget "https://zenodo.org/records/21064217/files/C2-lowres.tif?download=1" -O actin-lowres.tif
!wget "https://zenodo.org/records/21064217/files/C3-lowres.tif?download=1" -O bacteria-lowres.tif



dic_highres = io.imread("dic-highres.tif")
actin_highres = io.imread("actin-highres.tif")
bacteria_highres = io.imread("bacteria-highres.tif")

dic_lowres = io.imread("dic-lowres.tif")
actin_lowres = io.imread("actin-lowres.tif")
bacteria_lowres= io.imread("bacteria-lowres.tif")

print("✅ Data loaded successfully")


In [ ]:
# @title Step 7 - Take a look at your sample!

# Normalize helper
def normalize(img):
    img = img.astype(float)
    return (img - img.min()) / (img.max() - img.min() + 1e-8)


def auto_contrast(img, low=2, high=99.8):
    p_low, p_high = np.percentile(img, (low, high))
    img = np.clip(img, p_low, p_high)
    return (img - p_low) / (p_high - p_low + 1e-8)


# Define LUTs
green_lut = LinearSegmentedColormap.from_list("green", [(0,0,0),(0,1,0)])
red_lut   = LinearSegmentedColormap.from_list("red", [(0,0,0),(1,0,0)])

# Channels
lowres_channels = {
    "DIC": dic_lowres,
    "Actin": actin_lowres,
    "Bacteria": bacteria_lowres
}

def show_channel(channel):

    img = auto_contrast(lowres_channels[channel])

    # Central FoV (1/3)
    h, w = img.shape
    fov_h, fov_w = h // 3, w // 3
    cy, cx = h // 2, w // 2
    fov = img[
        cy - fov_h // 2 : cy + fov_h // 2,
        cx - fov_w // 2 : cx + fov_w // 2
    ]

    plt.figure(figsize=(5,5))

    if channel == "DIC":
        plt.imshow(fov, cmap='gray')
    elif channel == "Actin":
        plt.imshow(fov, cmap=red_lut)
    elif channel == "Bacteria":
        plt.imshow(fov, cmap=green_lut)

    plt.title(f"Central FoV: {channel}")
    plt.axis('off')
    plt.show()

widgets.interact(show_channel, channel=list(lowres_channels.keys()))


In [ ]:
# @title Step 8 — Acquire a low resolution overview image
# --- Pixel size (µm) ---
pixel_size = 1.4524  # µm per pixel

# --- Contrast ---
def auto_contrast(img, low=5, high=99):
    p_low, p_high = np.percentile(img, (low, high))
    img = np.clip(img, p_low, p_high)
    return (img - p_low) / (p_high - p_low + 1e-8)

# --- LUTs ---
green_lut = LinearSegmentedColormap.from_list("green", [(0,0,0),(0,1,0)])
red_lut   = LinearSegmentedColormap.from_list("red", [(0,0,0),(1,0,0)])

# --- Channels ---
lowres_channels = {
    "DIC": dic_lowres,
    "Actin": actin_lowres,
    "Bacteria": bacteria_lowres
}

# --- Main function ---
def tiled_view(channel, tiles):

    plt.close('all')

    img = auto_contrast(lowres_channels[channel])
    h, w = img.shape

    # Base FoV = 1/3 image
    base_h = h // 3
    base_w = w // 3

    # Tiled region
    region_h = base_h * tiles
    region_w = base_w * tiles

    cy, cx = h // 2, w // 2

    region = img[
        cy - region_h // 2 : cy + region_h // 2,
        cx - region_w // 2 : cx + region_w // 2
    ]

    # --- Convert to physical size ---
    size_y = region.shape[0] * pixel_size
    size_x = region.shape[1] * pixel_size

    # --- Plot ---
    plt.figure(figsize=(6,6))

    if channel == "DIC":
        plt.imshow(region, cmap='gray')
    elif channel == "Actin":
        plt.imshow(region, cmap=red_lut)
    elif channel == "Bacteria":
        plt.imshow(region, cmap=green_lut)

    plt.title(f"{tiles}×{tiles} tiles\nFoV: {size_x:.0f} × {size_y:.0f} µm")
    plt.axis('off')

    # --- Overlay tile grid ---
    base_h_px = region.shape[0] // tiles
    base_w_px = region.shape[1] // tiles

    for i in range(1, tiles):
        plt.axhline(i * base_h_px, color='white', linewidth=1)
        plt.axvline(i * base_w_px, color='white', linewidth=1)

    plt.show()

    print(f"🔬 Tiles acquired: {tiles*tiles}")
    print(f"📏 Field of view: {size_x:.1f} × {size_y:.1f} µm")


# --- Widgets ---
channel_widget = widgets.Dropdown(
    options=list(lowres_channels.keys()),
    value="DIC",
    description="Channel"
)

tiles_widget = widgets.Dropdown(
    options=[1,2,3],
    value=1,
    description="Tiles"
)

ui = widgets.VBox([channel_widget, tiles_widget])
out = widgets.interactive_output(
    tiled_view,
    {"channel": channel_widget, "tiles": tiles_widget}
)

display(ui, out)


#Step 9 — Analyse the image to find bacteria-cell interactions


In [ ]:
# @title Step 9.1: Segment cells
#Segment cells

def segment_cells_dic(dic_image, diameter=None, gpu=False):

    model = models.CellposeModel(model_type='cyto', gpu=gpu)

    masks, flows, styles = model.eval(
        dic_image,
        channels=[0, 0],  # single-channel mode
        diameter=diameter,
    )

    return masks, flows

masks, flows = segment_cells_dic(
    dic_lowres,
    diameter=None,
    gpu=True,
)



# Find contours
contours = measure.find_contours(masks, level=0.5)

plt.figure(figsize=(12,5))

# Raw image
plt.subplot(1,2,1)
plt.imshow(dic_lowres, cmap='gray')
plt.title("Raw DIC")
plt.axis('off')

# Overlay
plt.subplot(1,2,2)
plt.imshow(dic_lowres, cmap='gray')

for contour in contours:
    plt.plot(contour[:,1], contour[:,0], linewidth=1, color='red')

plt.title("Segmentation Boundaries")
plt.axis('off')
plt.show()


plt.show()


In [ ]:
# @title Step 9.2: Segment bacteria

# ----------------------------------------
# Bacteria segmentation and cell assignment
# ----------------------------------------

import numpy as np
import pandas as pd

from scipy.ndimage import distance_transform_edt

from skimage.measure import (
    label,
    regionprops,
    regionprops_table,
)

from skimage.morphology import remove_small_objects

# Pixel size of low resolution image
pixel_size = 1.4524  # µm/pixel


# ----------------------------------------
# Sigma-clipped segmentation
# ----------------------------------------

def sigma_clip_segmentation(
    image,
    sigma=3,
    min_size=3,
):

    vals = image.astype(np.float32)

    mu = vals.mean()
    sd = vals.std()

    thr = mu + sigma * sd

    mask = vals > thr

    mask = remove_small_objects(
        mask,
        min_size=min_size,
    )

    labels = label(mask)

    props = regionprops_table(
        labels,
        properties=[
            "label",
            "area",
            "centroid",
        ],
    )

    bacteria_df = pd.DataFrame(props)

    bacteria_df = bacteria_df.rename(
        columns={
            "label": "bacterium_id",
            "area": "area_px",
            "centroid-0": "centroid_y",
            "centroid-1": "centroid_x",
        }
    )

    bacteria_df["area_um2"] = (
        bacteria_df["area_px"]
        * pixel_size**2
    )

    return mask, labels, thr, bacteria_df


# ----------------------------------------
# Assign bacteria to cells
# ----------------------------------------

def analyze_bacteria(
    cell_labels,
    bacteria_labels,
):

    results = []

    for region in regionprops(bacteria_labels):

        bacterium_id = region.label

        y, x = region.centroid

        y = int(round(y))
        x = int(round(x))

        cell_id = cell_labels[y, x]

        if cell_id == 0:
            continue

        cell_mask = cell_labels == cell_id

        dist = distance_transform_edt(cell_mask)

        distance_inside = dist[y, x]

        results.append({
            "bacterium_id": bacterium_id,
            "cell_id": int(cell_id),
            "area_px": region.area,
            "area_um2": region.area * pixel_size**2,
            "centroid_x": x,
            "centroid_y": y,
            "distance_inside_px": distance_inside,
            "distance_inside_um": distance_inside * pixel_size,
        })

    return pd.DataFrame(results)


# ----------------------------------------
# Run analysis
# ----------------------------------------

bacteria_mask, bacteria_labels, threshold, bacteria_df = (
    sigma_clip_segmentation(
        bacteria_lowres,
        sigma=3,
        min_size=3,
    )
)

results_df = analyze_bacteria(
    masks,
    bacteria_labels,
)

# Cells containing bacteria
infected_cells = (
    results_df["cell_id"]
    .unique()
    .tolist()
)

overlay = np.zeros_like(masks)

for cid in infected_cells:
    overlay[masks == cid] = 1


# ----------------------------------------
# Statistics
# ----------------------------------------

print(f"Detected {len(bacteria_df)} bacteria")
print(f"Detected {len(infected_cells)} infected cells")
print(f"Threshold used: {threshold:.2f}")

display(
    bacteria_df[
        [
            "bacterium_id",
            "area_px",
            "area_um2",
        ]
    ].head()
)


# ----------------------------------------
# Visualization
# ----------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 6)
)

# ----------------------------------------
# Panel 1
# Bacteria segmentation
# ----------------------------------------

axes[0].imshow(
    auto_contrast(bacteria_lowres),
    cmap=green_lut,
)

for _, row in bacteria_df.iterrows():

    axes[0].text(
        row["centroid_x"],
        row["centroid_y"],
        str(int(row["bacterium_id"])),
        color="white",
        fontsize=6,
    )

axes[0].set_title(
    f"Bacteria Segmentation\nn={len(bacteria_df)}"
)
axes[0].axis("off")


# ----------------------------------------
# Panel 2
# Cells + bacteria
# ----------------------------------------

axes[1].imshow(
    auto_contrast(dic_lowres),
    cmap="gray",
)

axes[1].imshow(
    bacteria_mask,
    cmap=green_lut,
    alpha=0.6,
)

axes[1].imshow(
    masks,
    cmap="tab20",
    alpha=0.3,
)

axes[1].set_title(
    "Bacteria Assigned to Cells"
)
axes[1].axis("off")





# ----------------------------------------
# Distance information
# ----------------------------------------

display(
    results_df[
        [
            "bacterium_id",
            "cell_id",
            "distance_inside_um",
        ]
    ].head()
)

In [ ]:
# @title Step 9.3: Measure bacteria to cell distances and identify infected cells

# Global variables that will update automatically
selected_cells = []
target_positions = []

def select_bacteria(max_distance_um):

    global selected_cells
    global target_positions

    # Select bacteria close to the cell boundary
    selected_bacteria = results_df[
        results_df["distance_inside_um"] <= max_distance_um
    ]

    selected_cells = selected_bacteria["cell_id"].unique()

    # ----------------------------------------
    # Generate acquisition positions
    # ----------------------------------------

    target_positions = []

    for cid in selected_cells:

        cell_mask = masks == cid

        y, x = np.where(cell_mask)

        centroid_y = np.mean(y)
        centroid_x = np.mean(x)

        target_positions.append({
            "cell_id": int(cid),
            "x": float(centroid_x),
            "y": float(centroid_y),
        })

    # ----------------------------------------
    # Visualization
    # ----------------------------------------

    overlay = np.zeros_like(masks)

    for cid in selected_cells:
        overlay[masks == cid] = 1

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    # -----------------------
    # Selected bacteria
    # -----------------------

    axes[0].imshow(
        auto_contrast(bacteria_lowres),
        cmap=green_lut
    )

    for _, row in selected_bacteria.iterrows():

        axes[0].scatter(
            row["centroid_x"],
            row["centroid_y"],
            s=30,
            edgecolor="yellow",
            facecolor="none",
            linewidth=1.5,
        )

    axes[0].set_title(
        f"Bacteria ≤ {max_distance_um:.1f} µm from cell boundary\n"
        f"n = {len(selected_bacteria)}"
    )
    axes[0].axis("off")

    # -----------------------
    # Corresponding cells
    # -----------------------

    axes[1].imshow(
        auto_contrast(dic_lowres),
        cmap="gray"
    )

    axes[1].imshow(
        overlay,
        cmap="Reds",
        alpha=0.5
    )

    # acquisition positions
    for pos in target_positions:

        axes[1].plot(
            pos["x"],
            pos["y"],
            "yo",
            markersize=5,
        )

    axes[1].set_title(
        f"Selected cells\nn = {len(selected_cells)}"
    )
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    print(f"📍 Selected bacteria: {len(selected_bacteria)}")
    print(f"🧫 Selected cells: {len(selected_cells)}")



widgets.interact(
    select_bacteria,
    max_distance_um=widgets.FloatSlider(
        value=5,
        min=0,
        max=float(results_df["distance_inside_um"].max()),
        step=0.5,
        description="Distance (µm)"
    )
);


In [ ]:
# @title Step 10: Select infected cells for follow-up acquisition


# Global list of cells selected for revisiting
revisit_cells = []

def choose_revisit_cells(n_cells):

    global revisit_cells

    if len(target_positions) == 0:

        print(
            "⚠️ No target cells available. "
            "Adjust the bacteria distance threshold first."
        )

        revisit_cells = []
        return

    n_cells = min(
        n_cells,
        len(target_positions)
    )

    revisit_cells = random.sample(
        target_positions,
        n_cells
    )

    revisit_df = pd.DataFrame(revisit_cells)

    print(
        f"🎯 Selected {n_cells} cells for follow-up imaging"
    )

    display(revisit_df)

    # Visualize selected cells

    plt.figure(figsize=(6,6))

    plt.imshow(
        auto_contrast(dic_lowres),
        cmap="gray"
    )

    for cell in revisit_cells:

        plt.plot(
            cell["x"],
            cell["y"],
            "yo",
            markersize=8,
        )

        plt.text(
            cell["x"] + 5,
            cell["y"] + 5,
            str(cell["cell_id"]),
            color="yellow",
            fontsize=8,
        )

    plt.title(
        f"Cells selected for revisit (n={n_cells})"
    )

    plt.axis("off")
    plt.show()


widgets.interact(
    choose_revisit_cells,
    n_cells=widgets.IntSlider(
        value=min(5, max(1, len(target_positions))),
        min=1,
        max=max(1, len(target_positions)),
        step=1,
        description="Cells"
    )
);

In [ ]:
# Set channels
# @title Step 10: Select channels for targeted high-resolution acquisition

import ipywidgets as widgets
from IPython.display import display

# Global variable
selected_channels = []

# Available channels
channel_widgets = {
    "DIC": widgets.Checkbox(
        value=True,
        description="DIC"
    ),
    "Actin (AF647)": widgets.Checkbox(
        value=True,
        description="Actin (AF647)"
    ),
    "Bacteria (GFP)": widgets.Checkbox(
        value=True,
        description="Bacteria (GFP)"
    )
}


def update_channels(change=None):

    global selected_channels

    selected_channels = [
        name
        for name, widget in channel_widgets.items()
        if widget.value
    ]

    print("📷 Channels selected for acquisition:")
    for ch in selected_channels:
        print(f"   • {ch}")


# Attach callbacks
for widget in channel_widgets.values():
    widget.observe(update_channels, names="value")

# Display UI
display(
    widgets.VBox(
        [
            widgets.HTML(
                "<h4>Select channels for high-resolution acquisition</h4>"
            ),
            *channel_widgets.values()
        ]
    )
)

# Initialize
update_channels()

In [ ]:
# @title Step 11: Targeted High-Resolution Acquisition

from scipy.ndimage import find_objects

# High-res images
highres_channels = {
    "DIC": dic_highres,
    "Actin (AF647)": actin_highres,
    "Bacteria (GFP)": bacteria_highres,
}

# Scale factor between low-res and high-res
scale_factor = (
    dic_highres.shape[0] / dic_lowres.shape[0]
)

print("🚀 Starting targeted acquisition...")
print(f"Number of targets: {len(revisit_cells)}")
print(f"Channels: {selected_channels}")


for target in revisit_cells:

    cell_id = target["cell_id"]

    # -----------------------------
    # Find cell bounding box
    # -----------------------------

    cell_mask = masks == cell_id

    y, x = np.where(cell_mask)

    ymin = y.min()
    ymax = y.max()

    xmin = x.min()
    xmax = x.max()

    cell_height = ymax - ymin
    cell_width = xmax - xmin

    # -----------------------------
    # Expand bounding box by 2x
    # -----------------------------

    cy = (ymin + ymax) / 2
    cx = (xmin + xmax) / 2

    new_height = cell_height * 2
    new_width = cell_width * 2

    ymin = int(cy - new_height / 2)
    ymax = int(cy + new_height / 2)

    xmin = int(cx - new_width / 2)
    xmax = int(cx + new_width / 2)

    # keep inside image
    ymin = max(0, ymin)
    xmin = max(0, xmin)

    ymax = min(masks.shape[0], ymax)
    xmax = min(masks.shape[1], xmax)

    # -----------------------------
    # Convert to high-res coords
    # -----------------------------

    ymin_hr = int(ymin * scale_factor)
    ymax_hr = int(ymax * scale_factor)

    xmin_hr = int(xmin * scale_factor)
    xmax_hr = int(xmax * scale_factor)

    # -----------------------------
    # Display acquisition
    # -----------------------------

    n_channels = len(selected_channels)

    fig, axes = plt.subplots(
        1,
        n_channels,
        figsize=(5*n_channels, 5)
    )

    if n_channels == 1:
        axes = [axes]

    for ax, channel in zip(
        axes,
        selected_channels
    ):

        img = highres_channels[channel]

        crop = img[
            ymin_hr:ymax_hr,
            xmin_hr:xmax_hr
        ]

        if channel == "DIC":
            ax.imshow(
                auto_contrast(crop),
                cmap="gray"
            )

        elif channel == "Actin (AF647)":
            ax.imshow(
                auto_contrast(crop),
                cmap=red_lut
            )

        elif channel == "Bacteria (GFP)":
            ax.imshow(
                auto_contrast(crop),
                cmap=green_lut
            )

        ax.set_title(channel)
        ax.axis("off")

    plt.suptitle(
        f"Targeted Acquisition: Cell {cell_id}"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# @title Step 12: Efficiency Analysis and Reflection

# @markdown ----------------------------------------

# @markdown 🧠 Reflection



# @markdown • How much time, data storage and light exposure were saved by imaging only events of interest instead of all cells?"

# @markdown • In a 30-minute time series with 2-minute intervals, how much acquisition time, data storage and light exposure were saved by the event-driven strategy?

# @markdown • Would acquiring all cells provide substantially more biological information, or mostly generate additional data?

# @markdown • How might reduced light exposure improve cell health during long-term live-cell imaging?

# @markdown • How does targeted acquisition compare to simply randomly selecting cells for imaging?"

# @markdown ----------------------------------------


# ----------------------------------------
# Basic numbers
# ----------------------------------------

total_cells = len(np.unique(masks)) - 1
target_cells = len(selected_cells)

n_channels = len(selected_channels)

# ----------------------------------------
# Acquisition assumptions
# ----------------------------------------

# Approximate acquisition cost per cell and channel

highres_time_per_channel = 2.0      # seconds
highres_data_per_channel = 5.0      # MB

# ----------------------------------------
# Conventional acquisition
# ----------------------------------------

full_time = (
    total_cells
    * n_channels
    * highres_time_per_channel
)

full_data = (
    total_cells
    * n_channels
    * highres_data_per_channel
)

# ----------------------------------------
# Smart microscopy acquisition
# ----------------------------------------

targeted_time = (
    target_cells
    * n_channels
    * highres_time_per_channel
)

targeted_data = (
    target_cells
    * n_channels
    * highres_data_per_channel
)

# ----------------------------------------
# Savings
# ----------------------------------------

time_saved = full_time - targeted_time
data_saved = full_data - targeted_data

time_reduction = (
    100 * time_saved / full_time
    if full_time > 0 else 0
)

data_reduction = (
    100 * data_saved / full_data
    if full_data > 0 else 0
)

# Assume phototoxicity scales with
# the number of images acquired

light_reduction = time_reduction

# ----------------------------------------
# Time-series experiment
# ----------------------------------------

time_interval = 2      # min
total_duration = 30    # min

n_timepoints = total_duration // time_interval + 1

full_series_time = (
    full_time * n_timepoints
)

targeted_series_time = (
    targeted_time * n_timepoints
)

full_series_data = (
    full_data * n_timepoints
)

targeted_series_data = (
    targeted_data * n_timepoints
)

series_time_saved = (
    full_series_time
    - targeted_series_time
)

series_data_saved = (
    full_series_data
    - targeted_series_data
)

series_reduction = (
    100 * series_time_saved / full_series_time
    if full_series_time > 0 else 0
)

# ----------------------------------------
# Random acquisition comparison
# ----------------------------------------

random_cells = min(100, total_cells)

p_target = (
    target_cells / total_cells
    if total_cells > 0 else 0
)

expected_hits = random_cells * p_target

enrichment = (
    target_cells / expected_hits
    if expected_hits > 0 else np.inf
)

# ----------------------------------------
# Output
# ----------------------------------------

print("🔬 SMART MICROSCOPY EFFICIENCY ANALYSIS")
print("=" * 50)

print(f"Total cells detected:      {total_cells}")
print(f"Cells selected:           {target_cells}")
print(f"Channels acquired:         {n_channels}")

print()
print("📷 Conventional Acquisition")
print("-" * 30)
print(f"Time:                     {full_time:.1f} s")
print(f"Data volume:              {full_data:.1f} MB")

print()
print("🎯 Targeted Acquisition")
print("-" * 30)
print(f"Time:                     {targeted_time:.1f} s")
print(f"Data volume:              {targeted_data:.1f} MB")

print()
print("✅ SAVINGS")
print("-" * 30)
print(f"Time saved:               {time_saved:.1f} s")
print(f"Data saved:               {data_saved:.1f} MB")
print(f"Time reduction:           {time_reduction:.1f} %")
print(f"Data reduction:           {data_reduction:.1f} %")
print(f"Light exposure reduction: {light_reduction:.1f} %")

print()
print("📽️ TIME SERIES SCENARIO")
print("-" * 30)
print(
    f"{total_duration} min experiment "
    f"with {time_interval} min intervals"
)

print(f"Number of time points:    {n_timepoints}")


print()
print(f"Data saved:               {series_data_saved:.1f} MB")
print(f"Light exposure reduced:   {series_reduction:.1f} %")

print()
print("🎲 RANDOM SAMPLING COMPARISON")
print("-" * 30)

print(
    f"If {random_cells} cells were imaged randomly,"
)

print(
    f"the expected number of target cells would be "
    f"{expected_hits:.1f}."
)

print()

print(
    f"Targeted acquisition identified "
    f"{target_cells} target cells."
)

print(
    f"Enrichment over random sampling: "
    f"{enrichment:.1f}×"
)

# ----------------------------------------
# Visualization
# ----------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(14,4)
)

# Time

axes[0].bar(
    ["Full", "Targeted"],
    [full_time, targeted_time]
)

axes[0].set_title("Imaging Time")
axes[0].set_ylabel("Seconds")

# Data

axes[1].bar(
    ["Full", "Targeted"],
    [full_data, targeted_data]
)

axes[1].set_title("Data Volume")
axes[1].set_ylabel("MB")

# Cells

axes[2].bar(
    ["All Cells", "Revisited"],
    [total_cells, target_cells]
)

axes[2].set_title("Cells Imaged")
axes[2].set_ylabel("Number of Cells")

plt.tight_layout()
plt.show()

